# Configuration and Imports
This section sets up the necessary imports, constants, and configuration for the web scraper.

# Setup

In [1]:
import json
from bs4 import BeautifulSoup
import io
import pandas as pd
import requests
import re
import sqlite3
import datetime
import urllib
import time

from typing import List, Any, Dict, Generator

# Configuration constants
URL = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
DB_PATH = "../data/films.db"
JSON_EXPORT_PATH = "../web/films.json"

TARGET_TABLE_NAME = 'Highest-grossing films'

REQUEST_DELAY = 1.2

session = requests.Session()
session.headers.update(HEADERS)

# Data Models
Define dataclasses for structured data representation of films and revenue.

Create types for more convenient use

In [2]:
from dataclasses import dataclass

@dataclass
class FilmEntry:
    """Represents a film entry with key details."""
    title: str
    release_year: int
    director: List[str]  # Fixed typo from 'directior'
    box_office_revenue: int
    country: str
    url: str
    additional_data: Dict[str, Any]

# Data Cleaning Utilities
Functions to clean and parse raw data from Wikipedia, including removing references and parsing monetary values.

In [3]:
def remove_brackets_content(text: str) -> str:
    """
    Remove content within square brackets (e.g., references like [1]).
    
    Args:
        text (str): Input text.
    
    Returns:
        str: Text with brackets removed.
    """
    # Using regex to match any text between [ and ] (non-greedy)
    return re.sub(r'\[.*?\]', '', text)

def cleanup_table_refs(table: BeautifulSoup) -> None:
    """
    Clean up a BeautifulSoup table by removing superscripts and footnotes.
    
    Args:
        table (BeautifulSoup): The table to clean.
    """
    # Remove all indexes
    for i in table.find_all('sup'):
        i.decompose()
    # Remove all footnotes
    for text_node in table.find_all(string=True):
        if '†' in text_node:
            new_text = text_node.replace('†', '')
            text_node.replace_with(new_text)

def parse_money_to_dollars(text: str) -> List[float]:
    """
    Parse monetary values from text, handling ranges, scales, and qualifiers.
    Prioritizes 'gross' over 'net' if both are present.
    
    Args:
        text (str): Text containing money values.
    
    Returns:
        List[float]: List of parsed dollar values.
    """
    pattern = r'\$([\d,]+(?:\.\d+)?)\s*(?:-\s*([\d,]+(?:\.\d+)?))?\s*(million|billion|thousand)?\s*(?:\((gross|net)\))?'
    
    matches = re.finditer(pattern, text, re.IGNORECASE)
    
    parsed_data = []
    
    # Multipliers for scales
    multipliers = {
        'thousand': 1e3,
        'million': 1e6,
        'billion': 1e9,
        None: 1.0
    }
    
    for match in matches:
        start_str = match.group(1).replace(',', '')
        end_str = match.group(2)
        scale = match.group(3)
        qualifier = match.group(4)
        
        # Convert start number to float
        start_val = float(start_str)
        
        # Determine value based on range or single number
        if end_str:
            end_val = float(end_str.replace(',', ''))
            # Calculate mean for ranges
            base_value = (start_val + end_val) / 2
        else:
            base_value = start_val
            
        # Apply scale multiplier
        scale_factor = multipliers.get(scale.lower() if scale else None, 1.0)
        final_value = base_value * scale_factor
        final_value = int(round(final_value))
        
        # Normalize qualifier to lowercase for consistency
        qual_type = qualifier.lower() if qualifier else None
        
        parsed_data.append({
            'value': final_value,
            'type': qual_type
        })
    
    # Filtering Logic: Prioritize '(gross)' if it exists in the findings
    if not parsed_data:
        return []
    
    has_gross = any(item['type'] == 'gross' for item in parsed_data)
    
    if has_gross:
        # If any gross value exists, return only gross values
        result = [item['value'] for item in parsed_data if item['type'] == 'gross']
    else:
        # Otherwise, return all found values (net or unqualified)
        result = [item['value'] for item in parsed_data]
        
    return result

# Web Scraping
Functions to fetch and parse data from Wikipedia pages, including table extraction and individual film details.

## Scraping the Wikipedia Table

In [4]:
def get_tables() -> List[Dict[str, Any]]:
    """
    Fetch and filter tables from the Wikipedia page.
    
    Returns:
        List[Dict[str, Any]]: List of dicts with 'title' and 'table' keys.
    """
    print(f"Fetching data from {URL}...")
    response = session.get(URL, headers=HEADERS)
    response.raise_for_status()  # Ensure request was successful

    soup = BeautifulSoup(response.text, 'html.parser')
    body = soup.find('div', attrs={'id': 'bodyContent'})
    tables = body.find_all('table')
    tables = filter(lambda t: not t.find_parent('table'), tables)  # Find only tables that are not part of other tables
    tables = filter(lambda t: not t.find_parent('div', attrs={'role': 'navigation'}), tables)  # Remove navigation tables
    tables = map(lambda x: {
        'title': remove_brackets_content(x.find('caption', recursive=False).get_text()).strip(),
        'table': x
    }, tables)
    tables = list(tables)
    print('Found following tables: ', list(map(lambda x: x['title'], tables)))
    return tables

In [5]:
def get_film_links_from_table(table: BeautifulSoup) -> List[FilmEntry]:
    """
    Extract film links and basic info from the main table.
    
    Args:
        table (BeautifulSoup): The table to parse.
    
    Returns:
        List[FilmEntry]: List of FilmEntry objects with basic data.
    """
    base_url = 'https://en.wikipedia.org'
    
    cleanup_table_refs(table)
    df = pd.read_html(io.StringIO(str(table)), extract_links="body")[0]
    df['TitleLink'] = df['Title'].apply(lambda x: urllib.parse.urljoin(base_url, x[1]))
    df['Title'] = df['Title'].apply(lambda x: x[0])
    df['Year'] = df['Year'].apply(lambda x: x[0])
    df = df[['Title', 'TitleLink', 'Year']]

    return [FilmEntry(row['Title'], row['Year'], None, None, None, row['TitleLink'], {}) for _, row in df.iterrows()]

def fetch_film_data(entry: FilmEntry) -> pd.DataFrame:
    """
    Fetch detailed data from a film's Wikipedia page.
    
    Args:
        entry (FilmEntry): The film entry with URL.
    
    Returns:
        pd.DataFrame: DataFrame with parsed infobox data.
    """
    url = entry.url
    print(f"--Fetching data from {url}...")
    response = session.get(url, headers=HEADERS)
    response.raise_for_status()  # Ensure request was successful

    soup = BeautifulSoup(response.text, 'html.parser')

    body = soup.find('div', attrs={'id': 'mw-content-text'})
    table = body.find('table', attrs={'class': 'infobox vevent'})
    cleanup_table_refs(table)

    df = pd.DataFrame()

    for tr in table.find_all('tr'):
        header = tr.find('th')
        td = tr.find('td')

        if td and header:
            header = header.get_text()
            
            ls = td.find('ul') 
            if not ls:
                data = td.get_text()
            else:
                data = list(map(lambda x: x.get_text(), ls.find_all('li')))
            
            df[header] = [data]
    return df
 
def fetch_entry_data(entry: FilmEntry) -> None:
    """
    Populate a FilmEntry with detailed data from its page.
    
    Args:
        entry (FilmEntry): The entry to update.
    """
    df = fetch_film_data(entry)
    df.columns = df.columns.map(str.lower)

    entry.country = df.filter(regex='countr(y|ies)').iloc[0, 0]
    if type(entry.country) == list:
        entry.country = entry.country[0]

    box_office = df.filter(regex='box office').iloc[0, 0]  # Fixed variable name

    if type(box_office) == list:
        box_office = ' '.join(box_office)
        
    entry.box_office_revenue = parse_money_to_dollars(box_office)[0]

    entry.director = df.filter(regex='direct').iloc[0, 0]  # Fixed field name
    if type(entry.director) != list:
        entry.director = [entry.director] 
    entry.director = json.dumps(entry.director)

    return entry

In [6]:
def fetch_all_films() -> Generator[FilmEntry, None, None]:
    """
    Fetch all film entries from the target table.
    
    Returns:
        List[FilmEntry]: List of populated FilmEntry objects.
    """
    tables = get_tables()

    target_table = tables[[i for i, e in enumerate(tables) if e['title'].strip() == TARGET_TABLE_NAME][0]]['table']

    entries = get_film_links_from_table(target_table)
    for i, entry in enumerate(entries):
        while True:
            try:
                print(f'- {i})', end=' ')
                fetch_entry_data(entry)

                yield entry

                # Polite delay between requests to respect Wikipedia's rate limits
                # Wikipedia recommends ~1 request per second
                time.sleep(REQUEST_DELAY)
                break
            except requests.HTTPError as e:
                if e.response.status_code == 403:
                    print("-- -- 403 error, Retrying")
                    time.sleep(REQUEST_DELAY * 3)
                    continue
                else:
                    raise e


# Database Operations
Set up the database schema and functions to insert data.

In [7]:
SQL_INIT_QUERY = r'''
CREATE TABLE IF NOT EXISTS films (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    release_year INTEGER,
    director TEXT,
    box_office DECIMAL(12, 2),
    country TEXT,
    CONSTRAINT uc_film UNIQUE (title, release_year, director, country)
);
'''

with sqlite3.connect(DB_PATH) as db:
    db.execute(SQL_INIT_QUERY)

def insert_film_entry(entry: FilmEntry, db: sqlite3.Connection) -> None:
    """
    Insert or replace a film entry in the database.
    
    Args:
        entry (FilmEntry): The entry to insert.
        db (sqlite3.Connection): Database connection.
    """
    INSERT_QUERY = r'''
INSERT OR REPLACE INTO films(title, release_year, director, box_office, country) 
    VALUES(?, ?, ?, ?, ?);
'''

    db.execute(INSERT_QUERY, (entry.title, entry.release_year, entry.director, entry.box_office_revenue, entry.country))

# Execution and Export
Run the scraping process, store data in the database, and export to JSON.

In [8]:
with sqlite3.connect(DB_PATH) as db:
    for entry in fetch_all_films():
        try:
            insert_film_entry(entry, db)
        except Exception as e:
            print('-- --', "DB error: ", e)
        db.commit()

Fetching data from https://en.wikipedia.org/wiki/List_of_highest-grossing_films...
Found following tables:  ['Highest-grossing films', 'Highest-grossing films as of 2025 adjusted for inflation', 'High-grossing films by year of release', 'Timeline of the highest-grossing film record', 'Highest-grossing franchises and film series']
- 0) --Fetching data from https://en.wikipedia.org/wiki/Avatar_(2009_film)...
- 1) --Fetching data from https://en.wikipedia.org/wiki/Avengers:_Endgame...
- 2) --Fetching data from https://en.wikipedia.org/wiki/Avatar:_The_Way_of_Water...
- 3) --Fetching data from https://en.wikipedia.org/wiki/Titanic_(1997_film)...
- 4) --Fetching data from https://en.wikipedia.org/wiki/Ne_Zha_2...
- 5) --Fetching data from https://en.wikipedia.org/wiki/Star_Wars:_The_Force_Awakens...
- 6) --Fetching data from https://en.wikipedia.org/wiki/Avengers:_Infinity_War...
- 7) --Fetching data from https://en.wikipedia.org/wiki/Spider-Man:_No_Way_Home...
- 8) --Fetching data from htt

In [9]:
def export_sqlite_table_to_json(database_file: str, output_json_file: str) -> None:
    """
    Export the films table from SQLite to a JSON file.
    
    Args:
        database_file (str): Path to the SQLite database.
        output_json_file (str): Path to the output JSON file.
    """
    # Connect to the SQLite database
    with sqlite3.connect(database_file) as db:
        cursor = db.cursor()

        # Execute a query to select all data from the specified table
        cursor.execute(f"SELECT * FROM films")
        rows = cursor.fetchall()
        
        # Get column names from the cursor description
        columns = [description[0] for description in cursor.description]

        # Convert the rows to a list of dictionaries
        results = []
        for row in rows:
            results.append(dict(zip(columns, row)))

        # Convert to json format
        results = {
            'movies': results
        }

        # Write the list of dictionaries to a JSON file
        with open(output_json_file, 'w') as f:
            json.dump(results, f, indent=4)
            
        print(f"Successfully exported data from table 'films' to '{output_json_file}'")

export_sqlite_table_to_json(DB_PATH, JSON_EXPORT_PATH)

Successfully exported data from table 'films' to '../web/films.json'
